# 03 CleanData

Notebook นี้ใช้สำหรับ clean ข้อมูลต่อจากไฟล์ transformed

In [105]:
from pathlib import Path
import pandas as pd



## Load Transformed Data

เริ่มจากไฟล์ `vw_timestamp_dashboard_transformed.csv` ที่ได้จากขั้น transform

In [106]:
source_path = '../../data/interim/vw_timestamp_dashboard_transformed.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape before clean: {df.shape}')
df.head()


source: ../../data/interim/vw_timestamp_dashboard_transformed.csv
shape before clean: (32505, 40)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,0.0,2025-01-02 07:18:56,1.0,3.0,71-4711,SB1PL250102001,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",2025-01-02 07:19:02,0.0,...,0.0,NaN,NaN,2025-01-02 08:30:33,2025-01-02 08:30:33,NaN,NaN,NaN,NaN,NaN
1,SB1,0.0,2025-01-02 07:21:43,2.0,3.0,71-3545,SB1PL250102002,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",2025-01-02 07:21:46,0.0,...,0.0,NaN,NaN,2025-01-02 08:22:25,2025-01-02 08:32:32,NaN,NaN,NaN,NaN,NaN
2,SB1,0.0,2025-01-02 10:52:33,3.0,1.0,89-1359,SB1PL250102004,หจก.สำรวยเซรามิค,2025-01-02 10:52:39,0.0,...,0.0,NaN,NaN,2025-01-02 11:21:54,2025-01-02 11:23:32,2025-01-02 10:57:19,2025-01-02 10:59:32,NaN,NaN,NaN
3,SB1,0.0,2025-01-02 12:05:56,4.0,3.0,71-6663,SB1PL250102009,"SCGR Lumpoon Plant SCG Roofing Co., Ltd.",2025-01-02 12:05:59,0.0,...,0.0,NaN,NaN,NaN,NaN,2025-01-02 12:13:03,2025-01-02 12:14:35,NaN,NaN,NaN
4,SB1,0.0,2025-01-02 13:12:26,5.0,1.0,70-6399,SB1PL250102010,กรุงเทพฯ ดอนเมือง,2025-01-02 13:12:30,0.0,...,0.0,NaN,NaN,2025-01-02 13:24:44,2025-01-02 13:25:22,NaN,NaN,NaN,NaN,NaN


## Check Before Clean

เช็กคอลัมน์ `Unnamed: 39` และกระจายค่าของ `PackListStatus` ก่อนเริ่มลบ

In [107]:
print('has Unnamed: 39:', 'Unnamed: 39' in df.columns)
df['PackListStatus'].value_counts(dropna=False)

has Unnamed: 39: True


PackListStatus
OPERATORCOMPLETED    32497
Loading                  7
NaN                      1
Name: count, dtype: int64

## Step 1: ลบคอลัมน์ `Unnamed: 39` , `TruckOverTimeName` , `TruckOverTimeRemark`

ลบคอลัมน์ที่ไม่ต้องใช้ก่อน

In [108]:
df_clean = df.copy()

columns_to_drop = ['Unnamed: 39', 'TruckOverTimeName', 'TruckOverTimeRemark']
existing_columns_to_drop = [col for col in columns_to_drop if col in df_clean.columns]
df_clean = df_clean.drop(columns=existing_columns_to_drop)

print(f'dropped columns: {existing_columns_to_drop}')
print(f'shape after dropping: {df_clean.shape}')
df_clean.columns.tolist()[-10:]

dropped columns: ['Unnamed: 39', 'TruckOverTimeName', 'TruckOverTimeRemark']
shape after dropping: (32505, 37)


['PRESTIGEFittingSapAmount',
 'NEUSTILEFittingSapAmount',
 'DURAFittingSapAmount',
 'ACCESSORIESSapAmount',
 'TileStart',
 'TileEnd',
 'FittingStart',
 'FittingEnd',
 'AccStart',
 'AccEnd']

## Step 2: เก็บค่า `OPERATORCOMPLETED`

เอาเฉพาะแถวที่ `PackListStatus` เท่ากับ `OPERATORCOMPLETED`

In [109]:
rows_before = len(df_clean)
df_clean = df_clean.loc[df_clean['PackListStatus'] == 'OPERATORCOMPLETED'].copy()
rows_after_status = len(df_clean)

excluded_car_types = {'อื่นๆ', 'Unknown'}
car_type_series = df_clean['CarType'].fillna('').astype(str).str.strip()
car_type_keep_mask = ~car_type_series.isin(excluded_car_types)
df_clean = df_clean.loc[car_type_keep_mask].copy()
rows_after = len(df_clean)

print(f'rows after PackListStatus filter: {rows_after_status:,}')
print(f"rows removed by CarType in {sorted(excluded_car_types)}: {rows_after_status - rows_after:,}")
print(f'rows after all Step 2 filters: {rows_after:,}')
print(f'rows removed total in Step 2: {rows_before - rows_after:,}')

rows after PackListStatus filter: 32,497
rows removed by CarType in ['Unknown', 'อื่นๆ']: 0
rows after all Step 2 filters: 32,497
rows removed total in Step 2: 8


In [110]:
df_clean['PackListStatus'].value_counts(dropna=False)

PackListStatus
OPERATORCOMPLETED    32497
Name: count, dtype: int64

## Step 3: เก็บ Timestamp Columns

เก็บเฉพาะแถวที่ timestamp ทั้ง 5 ช่องไม่ว่าง อยู่วันเดียวกัน และค่าเวลาไม่ซ้ำกันในแถวเดียวกัน

In [111]:
required_timestamp_columns = [
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
]

timestamp_check = df_clean[required_timestamp_columns].copy()
for col in required_timestamp_columns:
    timestamp_check[col] = pd.to_datetime(timestamp_check[col], errors='coerce')

timestamp_check.isna().sum()

OperatorCarConfirm      0
CarConfirm              0
FirstPostPallet       327
LastPostPallet        327
PostingTime             9
dtype: int64

In [112]:
rows_before = len(df_clean)

for col in required_timestamp_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

non_null_mask = df_clean[required_timestamp_columns].notna().all(axis=1)
same_day_mask = df_clean.loc[non_null_mask, required_timestamp_columns].apply(
    lambda col: col.dt.date
).nunique(axis=1).eq(1)

same_day_index = df_clean.loc[non_null_mask].index[same_day_mask]
same_day_df = df_clean.loc[same_day_index].copy()
distinct_timestamp_mask = same_day_df[required_timestamp_columns].nunique(axis=1).eq(len(required_timestamp_columns))

df_clean = same_day_df.loc[distinct_timestamp_mask].copy()
rows_after = len(df_clean)

rows_after_non_null_same_day = len(same_day_df)
rows_removed_duplicate_timestamps = rows_after_non_null_same_day - rows_after

print(f'rows before timestamp filter: {rows_before:,}')
print(f'rows after non-null and same-day filter: {rows_after_non_null_same_day:,}')
print(f'rows removed because duplicate timestamps: {rows_removed_duplicate_timestamps:,}')
print(f'rows after timestamp filter: {rows_after:,}')
print(f'rows removed total: {rows_before - rows_after:,}')

rows before timestamp filter: 32,497
rows after non-null and same-day filter: 32,129
rows removed because duplicate timestamps: 5,480
rows after timestamp filter: 26,649
rows removed total: 5,848


## Step 4: Check Timestamp Order

ตรวจลำดับเวลาว่าต้องเรียงจาก `OperatorCarConfirm <= CarConfirm <= FirstPostPallet <= LastPostPallet <= PostingTime`

In [113]:
time_order_summary = pd.DataFrame(
    {
        'rule': [
            'OperatorCarConfirm <= CarConfirm',
            'CarConfirm <= FirstPostPallet',
            'FirstPostPallet <= LastPostPallet',
            'LastPostPallet <= PostingTime',
        ],
        'violated_rows': [
            (df_clean['OperatorCarConfirm'] > df_clean['CarConfirm']).sum(),
            (df_clean['CarConfirm'] > df_clean['FirstPostPallet']).sum(),
            (df_clean['FirstPostPallet'] > df_clean['LastPostPallet']).sum(),
            (df_clean['LastPostPallet'] > df_clean['PostingTime']).sum(),
        ],
    }
)

time_order_summary

,rule,violated_rows
0,OperatorCarConfirm <= CarConfirm,111
1,CarConfirm <= FirstPostPallet,0
2,FirstPostPallet <= LastPostPallet,0
3,LastPostPallet <= PostingTime,0


## Step 5: Remove Negative Durations

คำนวณ duration ชั่วคราวเพื่อตรวจแถวที่เวลาติดลบ แล้วลบแถวนั้นออก

In [114]:
duration_check = pd.DataFrame(index=df_clean.index)
duration_check['wait_call_min'] = (df_clean['CarConfirm'] - df_clean['OperatorCarConfirm']).dt.total_seconds() / 60
duration_check['prepare_loading_min'] = (df_clean['FirstPostPallet'] - df_clean['CarConfirm']).dt.total_seconds() / 60
duration_check['loading_time_min'] = (df_clean['LastPostPallet'] - df_clean['FirstPostPallet']).dt.total_seconds() / 60
duration_check['close_job_min'] = (df_clean['PostingTime'] - df_clean['LastPostPallet']).dt.total_seconds() / 60

negative_duration_summary = pd.DataFrame(
    {
        'duration_name': duration_check.columns,
        'negative_rows': [(duration_check[col] < 0).sum() for col in duration_check.columns],
    }
)

negative_duration_summary

,duration_name,negative_rows
0,wait_call_min,111
1,prepare_loading_min,0
2,loading_time_min,0
3,close_job_min,0


In [115]:
rows_before = len(df_clean)
negative_duration_mask = (duration_check < 0).any(axis=1)
df_clean = df_clean.loc[~negative_duration_mask].copy()
rows_after = len(df_clean)

print(f'rows before negative duration filter: {rows_before:,}')
print(f'rows after negative duration filter: {rows_after:,}')
print(f'rows removed: {rows_before - rows_after:,}')

rows before negative duration filter: 26,649
rows after negative duration filter: 26,538
rows removed: 111


## Step 6: กรองข้อมูลเฉพาะปี 2025 เป็นต้นไป

เก็บเฉพาะแถวที่ `OperatorCarConfirm` อยู่ในปี 2025 หรือหลังจากนั้น เนื่องจากข้อมูลก่อนหน้านี้สะท้อนพฤติกรรมระบบเก่าที่ต่างจากปัจจุบันอย่างมีนัยสำคัญ (mean total_time ลดจาก ~250 min ในปี 2022 เหลือ ~60 min ในปี 2025)

In [116]:
rows_before = len(df_clean)

year_mask = df_clean['OperatorCarConfirm'].dt.year >= 2025
df_clean = df_clean.loc[year_mask].copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

pct_removed = (rows_removed / rows_before * 100) if rows_before > 0 else 0

print(f'rows before year filter: {rows_before:,}')
print(f'rows after year filter (2025+): {rows_after:,}')
print(f'rows removed (before 2025): {rows_removed:,} ({pct_removed:.1f}%)')
print()
print('Year distribution after filter:')
print(df_clean['OperatorCarConfirm'].dt.year.value_counts().sort_index().to_string())

rows before year filter: 26,538
rows after year filter (2025+): 26,538
rows removed (before 2025): 0 (0.0%)

Year distribution after filter:
OperatorCarConfirm
2025    19344
2026     7194


## Step 7: Removal Summary

สรุปจำนวนแถวที่ถูกตัดออกแยกตามเหตุผลที่ตรวจในขั้นนี้

In [117]:
removal_summary = pd.DataFrame(
    {
        'reason': [
            'OperatorCarConfirm > CarConfirm',
            'CarConfirm > FirstPostPallet',
            'FirstPostPallet > LastPostPallet',
            'LastPostPallet > PostingTime',
            'wait_call_min < 0',
            'prepare_loading_min < 0',
            'loading_time_min < 0',
            'close_job_min < 0',
            'any negative duration row removed',
            'OperatorCarConfirm year < 2025',
        ],
        'rows': [
            int(time_order_summary.loc[time_order_summary['rule'] == 'OperatorCarConfirm <= CarConfirm', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'CarConfirm <= FirstPostPallet', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'FirstPostPallet <= LastPostPallet', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'LastPostPallet <= PostingTime', 'violated_rows'].iloc[0]),
            int((duration_check['wait_call_min'] < 0).sum()),
            int((duration_check['prepare_loading_min'] < 0).sum()),
            int((duration_check['loading_time_min'] < 0).sum()),
            int((duration_check['close_job_min'] < 0).sum()),
            int(negative_duration_mask.sum()),
            int(rows_removed),
        ],
    }
)

removal_summary

,reason,rows
0,OperatorCarConfirm > CarConfirm,111
1,CarConfirm > FirstPostPallet,0
2,FirstPostPallet > LastPostPallet,0
3,LastPostPallet > PostingTime,0
4,wait_call_min < 0,111
5,prepare_loading_min < 0,0
6,loading_time_min < 0,0
7,close_job_min < 0,0
8,any negative duration row removed,111
9,OperatorCarConfirm year < 2025,0


## Preview Cleaned Data

ดูข้อมูลหลัง clean ขั้นล่าสุด

In [118]:
print(f'final shape for current step: {df_clean.shape}')
df_clean[['OperatorCarConfirm', 'CarConfirm', 'FirstPostPallet', 'LastPostPallet', 'PostingTime', 'PackListStatus']].head(10)

final shape for current step: (26538, 37)


,OperatorCarConfirm,CarConfirm,FirstPostPallet,LastPostPallet,PostingTime,PackListStatus
0,2025-01-02 07:18:57,2025-01-02 07:21:03,2025-01-02 07:29:10,2025-01-02 07:46:01,2025-01-02 08:31:00,OPERATORCOMPLETED
1,2025-01-02 07:21:45,2025-01-02 07:37:48,2025-01-02 07:40:33,2025-01-02 08:07:36,2025-01-02 08:43:43,OPERATORCOMPLETED
2,2025-01-02 10:52:36,2025-01-02 11:13:57,2025-01-02 11:19:37,2025-01-02 11:24:31,2025-01-02 11:32:59,OPERATORCOMPLETED
3,2025-01-02 12:05:57,2025-01-02 12:12:50,2025-01-02 12:14:46,2025-01-02 12:22:01,2025-01-02 12:43:57,OPERATORCOMPLETED
4,2025-01-02 13:12:27,2025-01-02 13:14:18,2025-01-02 13:19:06,2025-01-02 13:29:59,2025-01-02 13:41:07,OPERATORCOMPLETED
5,2025-01-02 13:13:09,2025-01-02 13:29:03,2025-01-02 13:29:32,2025-01-02 13:46:20,2025-01-02 13:58:21,OPERATORCOMPLETED
6,2025-01-02 13:16:44,2025-01-02 13:20:43,2025-01-02 13:32:28,2025-01-02 13:41:20,2025-01-02 13:56:48,OPERATORCOMPLETED
7,2025-01-02 13:39:11,2025-01-02 13:51:51,2025-01-02 13:52:05,2025-01-02 14:22:48,2025-01-02 14:37:57,OPERATORCOMPLETED
8,2025-01-02 13:40:04,2025-01-02 13:43:53,2025-01-02 13:48:19,2025-01-02 13:54:22,2025-01-02 14:15:17,OPERATORCOMPLETED
9,2025-01-02 14:22:33,2025-01-02 14:23:32,2025-01-02 14:43:00,2025-01-02 14:53:41,2025-01-02 15:10:53,OPERATORCOMPLETED


## Save Current Clean File

บันทึกผล clean ปัจจุบันไว้ก่อน เพื่อใช้ต่อในขั้นถัดไป

In [119]:
output_path = '../../data/interim/vw_timestamp_dashboard_clean.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')
output_path


'../../data/interim/vw_timestamp_dashboard_clean.csv'